# 03 User Funnel Analysis

This notebook is the third step of the **AI-Powered Lead Generation & Recommendation Analytics Platform** project.

The goal of this stage is to create and analyse a simulated user interaction event log for a live-commerce and e-commerce lead generation context.

The original Olist dataset contains real orders, products, sellers, customers, payments, and reviews, but it does not contain clickstream events such as product views, clicks, inquiries, or add-to-cart actions.

To make the project closer to a TikTok LIVE / SMB lead generation scenario, this notebook creates a simulated event log with the following event types:

1. `view`
2. `click`
3. `add_to_cart`
4. `inquiry`
5. `purchase`

The simulated event log will be used to analyse user conversion funnels, lead quality, category-level conversion, traffic source performance, and seller-level lead generation performance.

## Step 1: Import Libraries and Define Project Paths

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set random seed for reproducibility
np.random.seed(42)

# Define project paths
BASE_DIR = Path("..")

PROCESSED_DIR = BASE_DIR / "data" / "processed"
FINAL_DIR = BASE_DIR / "data" / "final"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLE_OUTPUT_DIR = OUTPUT_DIR / "tables"

# Create folders if they do not exist
FINAL_DIR.mkdir(parents=True, exist_ok=True)
TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries imported and project paths defined successfully.")

Libraries imported and project paths defined successfully.


## Step 2: Load Cleaned Order-Level Dataset

In [11]:
# Load cleaned order-level dataset
order_base = pd.read_csv(PROCESSED_DIR / "clean_order_base.csv")

# Convert order_time to datetime
order_base["order_time"] = pd.to_datetime(order_base["order_time"])

# Preview dataset
order_base.head()

,order_id,customer_id,user_id,product_id,seller_id,order_time,order_status,price,freight_value,payment_value,...,review_comment_message,order_delivered_customer_date,order_estimated_delivery_date,order_date,order_year,order_month,gmv,delivery_delay_days,late_delivery_flag,price_band
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-13 08:59:02,delivered,58.90,13.29,72.19,...,"Perfeito, produto entregue antes do combinado.",2017-09-20 23:43:48,2017-09-29,2017-09-13,2017,2017-09,72.19,-9.0,0,Medium
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-04-26 10:53:06,delivered,239.90,19.93,259.83,...,NaN,2017-05-12 16:04:24,2017-05-15,2017-04-26,2017,2017-04,259.83,-3.0,0,Premium
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-14 14:33:31,delivered,199.00,17.87,216.87,...,Chegou antes do prazo previsto e o produto sur...,2018-01-22 13:19:16,2018-02-05,2018-01-14,2018,2018-01,216.87,-14.0,0,Premium
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-08 10:00:35,delivered,12.99,12.79,25.78,...,NaN,2018-08-14 13:32:39,2018-08-20,2018-08-08,2018,2018-08,25.78,-6.0,0,Low
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-04 13:57:51,delivered,199.90,18.14,218.04,...,Gostei pois veio no prazo determinado .,2017-03-01 16:42:31,2017-03-17,2017-02-04,2017,2017-02,218.04,-16.0,0,Premium


## Step 3: Inspect Input Dataset

In [12]:
print("Dataset shape:", order_base.shape)

key_columns = [
    "order_id", "user_id", "product_id", "seller_id", "order_time",
    "order_status", "category", "price", "gmv", "review_score",
    "price_band", "late_delivery_flag"
]

order_base[key_columns].head()

Dataset shape: (112650, 32)


,order_id,user_id,product_id,seller_id,order_time,order_status,category,price,gmv,review_score,price_band,late_delivery_flag
0,00010242fe8c5a6d1ba2dd792cb16214,871766c5855e863f6eccc05f988b23cb,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-13 08:59:02,delivered,cool_stuff,58.90,72.19,5.0,Medium,0
1,00018f77f2f0320c557190d7a144bdd3,eb28e67c4c0b83846050ddfb8a35d051,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-04-26 10:53:06,delivered,pet_shop,239.90,259.83,4.0,Premium,0
2,000229ec398224ef6ca0657da4fc703e,3818d81c6709e39d06b2738a8d3a2474,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-14 14:33:31,delivered,furniture_decor,199.00,216.87,5.0,Premium,0
3,00024acbcdf0a6daa1e931b038114c75,af861d436cfc08b2c2ddefd0ba074622,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-08 10:00:35,delivered,perfumery,12.99,25.78,4.0,Low,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,64b576fb70d441e8f1b2d7d446e483c5,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-04 13:57:51,delivered,garden_tools,199.90,218.04,5.0,Premium,0


## Step 4: Filter Delivered Orders as Purchase Anchors

In the Olist dataset, real purchases are represented by orders.

For the simulated funnel, I use delivered orders as confirmed purchase events.

Each delivered order will become a `purchase` event, and earlier simulated events such as `view`, `click`, `add_to_cart`, and `inquiry` will be generated before the purchase timestamp.

This creates a realistic user journey:

`view → click → add_to_cart → inquiry → purchase`

In [13]:
# Keep delivered orders as confirmed purchase anchors
purchase_orders = order_base[
    order_base["order_status"] == "delivered"
].copy()

print("Delivered purchase records:", purchase_orders.shape)

purchase_orders[[
    "order_id", "user_id", "product_id", "seller_id",
    "order_time", "category", "price", "gmv", "review_score"
]].head()

Delivered purchase records: (110197, 32)


,order_id,user_id,product_id,seller_id,order_time,category,price,gmv,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,871766c5855e863f6eccc05f988b23cb,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-13 08:59:02,cool_stuff,58.90,72.19,5.0
1,00018f77f2f0320c557190d7a144bdd3,eb28e67c4c0b83846050ddfb8a35d051,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-04-26 10:53:06,pet_shop,239.90,259.83,4.0
2,000229ec398224ef6ca0657da4fc703e,3818d81c6709e39d06b2738a8d3a2474,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-14 14:33:31,furniture_decor,199.00,216.87,5.0
3,00024acbcdf0a6daa1e931b038114c75,af861d436cfc08b2c2ddefd0ba074622,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-08 10:00:35,perfumery,12.99,25.78,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,64b576fb70d441e8f1b2d7d446e483c5,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-04 13:57:51,garden_tools,199.90,218.04,5.0


## Step 5: Create User Historical Features

Before simulating behaviour events, I create simple user-level historical features. These features help make the event simulation more business-driven rather than completely random.

User features include:

- Total historical orders
- Total GMV
- Average order value
- Average review score

These fields will later support user segmentation and lead quality analysis.

In [14]:
# Create user-level historical features
user_features = purchase_orders.groupby("user_id").agg(
    user_total_orders=("order_id", "nunique"),
    user_total_gmv=("gmv", "sum"),
    user_avg_order_value=("gmv", "mean"),
    user_avg_review_score=("review_score", "mean")
).reset_index()

# Create user value segment based on total GMV
user_features["user_value_segment"] = pd.qcut(
    user_features["user_total_gmv"],
    q=3,
    labels=["Low Value", "Medium Value", "High Value"],
    duplicates="drop"
)

user_features.head()

,user_id,user_total_orders,user_total_gmv,user_avg_order_value,user_avg_review_score,user_value_segment
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,5.0,Medium Value
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,4.0,Low Value
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,3.0,Medium Value
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,4.0,Low Value
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,5.0,High Value


## Step 6: Create Seller Quality Features

Seller quality is important in a lead generation and recommendation context. A seller with stronger review scores and higher order volume may be more suitable for recommendation exposure because users are more likely to trust the seller.

Seller features include:

- Total seller orders
- Total seller GMV
- Average seller review score
- Late delivery rate

In [15]:
# Create seller-level quality features
seller_features = purchase_orders.groupby("seller_id").agg(
    seller_total_orders=("order_id", "nunique"),
    seller_total_gmv=("gmv", "sum"),
    seller_avg_review_score=("review_score", "mean"),
    seller_late_delivery_rate=("late_delivery_flag", "mean")
).reset_index()

# Fill missing values if any
seller_features["seller_late_delivery_rate"] = seller_features["seller_late_delivery_rate"].fillna(0)

seller_features.head()

,seller_id,seller_total_orders,seller_total_gmv,seller_avg_review_score,seller_late_delivery_rate
0,0015a82c2db000af6aaaf3ae2ecb0532,3,2748.06,3.666667,0.000000
1,001cca7ae9ae17fb1caed9dfb1094831,195,33142.90,3.978632,0.051282
2,002100f778ceb8431b7a1020ff7ab48f,50,1995.16,4.037037,0.166667
3,003554e2dce176b5555353e4f3555ac8,1,139.38,5.000000,0.000000
4,004c9cd9d87a3c30c522c48c4fc07416,156,23094.02,4.166667,0.059524


## Step 7: Merge User and Seller Features Back to Purchase Orders

I merge user-level and seller-level features back to each delivered order.

This enriched purchase dataset will be used to simulate more realistic upstream user events.

In [16]:
# Merge user and seller features
purchase_enriched = (
    purchase_orders
    .merge(user_features, on="user_id", how="left")
    .merge(seller_features, on="seller_id", how="left")
)

purchase_enriched.head()

,order_id,customer_id,user_id,product_id,seller_id,order_time,order_status,price,freight_value,payment_value,...,price_band,user_total_orders,user_total_gmv,user_avg_order_value,user_avg_review_score,user_value_segment,seller_total_orders,seller_total_gmv,seller_avg_review_score,seller_late_delivery_rate
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-13 08:59:02,delivered,58.90,13.29,72.19,...,Medium,1,72.19,72.19,5.0,Low Value,134,14662.01,4.129252,0.068027
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-04-26 10:53:06,delivered,239.90,19.93,259.83,...,Premium,2,284.56,142.28,4.5,High Value,121,12025.18,3.788732,0.112676
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-14 14:33:31,delivered,199.00,17.87,216.87,...,Premium,1,216.87,216.87,5.0,High Value,12,3548.95,3.785714,0.000000
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-08 10:00:35,delivered,12.99,12.79,25.78,...,Low,1,25.78,25.78,4.0,Low Value,13,1331.87,3.750000,0.062500
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-04 13:57:51,delivered,199.90,18.14,218.04,...,Premium,1,218.04,218.04,5.0,High Value,26,3999.89,3.925926,0.074074


## Step 8: Define Event Simulation Logic

Since the original dataset does not contain real clickstream data, I simulate user interaction events based on business assumptions.

The logic is:

1. Every delivered order has one `purchase` event.
2. Every purchase should have at least one earlier `view` event.
3. Click, add-to-cart, and inquiry events are generated probabilistically.
4. The probability of stronger engagement increases when:
   - Product review score is higher;
   - User has higher historical value;
   - Seller review score is higher;
   - Product is not extremely expensive.

This makes the simulated event log more realistic than pure random generation.

In [18]:
def get_engagement_multiplier(row):
    """
    Create an engagement multiplier based on product, user, seller, and price signals.
    
    The multiplier is used to make simulated user behaviour more realistic.
    Higher values indicate stronger user engagement potential.
    """
    multiplier = 1.0
    
    # Higher product review score increases engagement
    if row["review_score"] >= 4:
        multiplier += 0.20
    elif row["review_score"] <= 2:
        multiplier -= 0.20
    
    # High-value users are more likely to engage
    if row["user_value_segment"] == "High Value":
        multiplier += 0.15
    elif row["user_value_segment"] == "Low Value":
        multiplier -= 0.05
    
    # Better seller quality increases engagement
    if row["seller_avg_review_score"] >= 4:
        multiplier += 0.10
    elif row["seller_avg_review_score"] <= 2.5:
        multiplier -= 0.10
    
    # Premium products may have lower conversion due to price sensitivity
    if row["price_band"] == "Premium":
        multiplier -= 0.10
    elif row["price_band"] == "Low":
        multiplier += 0.05
    
    # Keep multiplier within a reasonable range
    return max(0.5, min(multiplier, 1.5))

## Step 9: Simulate User Event Log

For each purchase record, I generate a user journey.

Possible event sequence:

- `view`
- `click`
- `add_to_cart`
- `inquiry`
- `purchase`

Some users may only view and purchase directly, while others may go through more intermediate steps.

I also randomly assign:

- Device type
- Traffic source
- Session ID

These fields help support funnel and attribution-style analysis.

In [19]:
event_records = []

device_types = ["iOS", "Android", "Web"]
traffic_sources = ["For You Feed", "Live Stream", "Search", "Shop Tab", "Creator Profile"]

for idx, row in purchase_enriched.iterrows():
    engagement_multiplier = get_engagement_multiplier(row)
    
    # Base probabilities adjusted by engagement multiplier
    click_prob = min(0.85, 0.55 * engagement_multiplier)
    add_to_cart_prob = min(0.75, 0.40 * engagement_multiplier)
    inquiry_prob = min(0.65, 0.28 * engagement_multiplier)
    
    user_id = row["user_id"]
    product_id = row["product_id"]
    seller_id = row["seller_id"]
    category = row["category"]
    purchase_time = row["order_time"]
    order_id = row["order_id"]
    
    session_id = f"session_{idx}"
    device_type = np.random.choice(device_types, p=[0.42, 0.43, 0.15])
    traffic_source = np.random.choice(
        traffic_sources,
        p=[0.35, 0.25, 0.18, 0.14, 0.08]
    )
    
    # Generate event timestamps before purchase
    view_time = purchase_time - pd.Timedelta(minutes=np.random.randint(30, 240))
    click_time = view_time + pd.Timedelta(minutes=np.random.randint(1, 20))
    add_to_cart_time = click_time + pd.Timedelta(minutes=np.random.randint(1, 30))
    inquiry_time = add_to_cart_time + pd.Timedelta(minutes=np.random.randint(1, 40))
    
    # Every purchase journey starts with a view event
    event_records.append({
        "user_id": user_id,
        "session_id": session_id,
        "event_time": view_time,
        "event_type": "view",
        "order_id": order_id,
        "product_id": product_id,
        "seller_id": seller_id,
        "category": category,
        "device_type": device_type,
        "traffic_source": traffic_source,
        "price": row["price"],
        "gmv": 0,
        "review_score": row["review_score"],
        "user_value_segment": row["user_value_segment"]
    })
    
    has_click = np.random.rand() < click_prob
    has_add_to_cart = np.random.rand() < add_to_cart_prob
    has_inquiry = np.random.rand() < inquiry_prob
    
    if has_click:
        event_records.append({
            "user_id": user_id,
            "session_id": session_id,
            "event_time": click_time,
            "event_type": "click",
            "order_id": order_id,
            "product_id": product_id,
            "seller_id": seller_id,
            "category": category,
            "device_type": device_type,
            "traffic_source": traffic_source,
            "price": row["price"],
            "gmv": 0,
            "review_score": row["review_score"],
            "user_value_segment": row["user_value_segment"]
        })
    
    if has_add_to_cart:
        event_records.append({
            "user_id": user_id,
            "session_id": session_id,
            "event_time": add_to_cart_time,
            "event_type": "add_to_cart",
            "order_id": order_id,
            "product_id": product_id,
            "seller_id": seller_id,
            "category": category,
            "device_type": device_type,
            "traffic_source": traffic_source,
            "price": row["price"],
            "gmv": 0,
            "review_score": row["review_score"],
            "user_value_segment": row["user_value_segment"]
        })
    
    if has_inquiry:
        event_records.append({
            "user_id": user_id,
            "session_id": session_id,
            "event_time": inquiry_time,
            "event_type": "inquiry",
            "order_id": order_id,
            "product_id": product_id,
            "seller_id": seller_id,
            "category": category,
            "device_type": device_type,
            "traffic_source": traffic_source,
            "price": row["price"],
            "gmv": 0,
            "review_score": row["review_score"],
            "user_value_segment": row["user_value_segment"]
        })
    
    # Purchase event
    event_records.append({
        "user_id": user_id,
        "session_id": session_id,
        "event_time": purchase_time,
        "event_type": "purchase",
        "order_id": order_id,
        "product_id": product_id,
        "seller_id": seller_id,
        "category": category,
        "device_type": device_type,
        "traffic_source": traffic_source,
        "price": row["price"],
        "gmv": row["gmv"],
        "review_score": row["review_score"],
        "user_value_segment": row["user_value_segment"]
    })

# Convert event records into DataFrame
fact_user_events = pd.DataFrame(event_records)

# Sort events by time
fact_user_events = fact_user_events.sort_values("event_time").reset_index(drop=True)

fact_user_events.head(10)

,user_id,session_id,event_time,event_type,order_id,product_id,seller_id,category,device_type,traffic_source,price,gmv,review_score,user_value_segment
0,830d5b7aaa3b6f1e9ad63703bec97d23,session_82573,2016-09-15 09:18:38,view,bfbd0f9bdef84302105ad712db648a6c,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,health_beauty,iOS,Live Stream,44.99,0.00,1.0,Medium Value
1,830d5b7aaa3b6f1e9ad63703bec97d23,session_82572,2016-09-15 09:21:38,view,bfbd0f9bdef84302105ad712db648a6c,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,health_beauty,Android,Creator Profile,44.99,0.00,1.0,Medium Value
2,830d5b7aaa3b6f1e9ad63703bec97d23,session_82572,2016-09-15 09:31:38,click,bfbd0f9bdef84302105ad712db648a6c,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,health_beauty,Android,Creator Profile,44.99,0.00,1.0,Medium Value
3,830d5b7aaa3b6f1e9ad63703bec97d23,session_82573,2016-09-15 09:33:38,add_to_cart,bfbd0f9bdef84302105ad712db648a6c,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,health_beauty,iOS,Live Stream,44.99,0.00,1.0,Medium Value
4,830d5b7aaa3b6f1e9ad63703bec97d23,session_82573,2016-09-15 10:02:38,inquiry,bfbd0f9bdef84302105ad712db648a6c,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,health_beauty,iOS,Live Stream,44.99,0.00,1.0,Medium Value
5,830d5b7aaa3b6f1e9ad63703bec97d23,session_82574,2016-09-15 10:05:38,view,bfbd0f9bdef84302105ad712db648a6c,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,health_beauty,Android,Shop Tab,44.99,0.00,1.0,Medium Value
6,830d5b7aaa3b6f1e9ad63703bec97d23,session_82574,2016-09-15 10:33:38,add_to_cart,bfbd0f9bdef84302105ad712db648a6c,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,health_beauty,Android,Shop Tab,44.99,0.00,1.0,Medium Value
7,830d5b7aaa3b6f1e9ad63703bec97d23,session_82573,2016-09-15 12:16:38,purchase,bfbd0f9bdef84302105ad712db648a6c,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,health_beauty,iOS,Live Stream,44.99,47.82,1.0,Medium Value
8,830d5b7aaa3b6f1e9ad63703bec97d23,session_82572,2016-09-15 12:16:38,purchase,bfbd0f9bdef84302105ad712db648a6c,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,health_beauty,Android,Creator Profile,44.99,47.82,1.0,Medium Value
9,830d5b7aaa3b6f1e9ad63703bec97d23,session_82574,2016-09-15 12:16:38,purchase,bfbd0f9bdef84302105ad712db648a6c,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,health_beauty,Android,Shop Tab,44.99,47.82,1.0,Medium Value


## Step 10: Validate Simulated Event Log

After generating the event log, we validate its structure.

I check:

- Total number of events
- Event type distribution
- Number of unique users
- Number of unique sessions
- Date range of events

This ensures that the simulated event log is reasonable before using it for funnel analysis.

In [20]:
print("Event log shape:", fact_user_events.shape)
print("Unique users:", fact_user_events["user_id"].nunique())
print("Unique sessions:", fact_user_events["session_id"].nunique())
print("Date range:", fact_user_events["event_time"].min(), "to", fact_user_events["event_time"].max())

event_distribution = fact_user_events["event_type"].value_counts().reset_index()
event_distribution.columns = ["event_type", "event_count"]

event_distribution

Event log shape: (386012, 14)
Unique users: 93358
Unique sessions: 110197
Date range: 2016-09-15 09:18:38 to 2018-08-29 15:00:37


,event_type,event_count
0,view,110197
1,purchase,110197
2,click,74013
3,add_to_cart,53850
4,inquiry,37755


## Step 11: Overall Funnel Summary

This step calculates the overall conversion funnel.

Metrics include:

- Number of users at each event stage
- Conversion rate from view to each downstream event

This helps identify where users drop off in the lead generation journey.

In [21]:
# Define funnel order
funnel_order = ["view", "click", "add_to_cart", "inquiry", "purchase"]

# Count unique users at each funnel stage
funnel_summary = (
    fact_user_events
    .groupby("event_type")["user_id"]
    .nunique()
    .reindex(funnel_order)
    .reset_index()
)

funnel_summary.columns = ["event_type", "unique_users"]

# Calculate conversion rate relative to view stage
view_users = funnel_summary.loc[
    funnel_summary["event_type"] == "view", 
    "unique_users"
].iloc[0]

funnel_summary["conversion_from_view"] = (
    funnel_summary["unique_users"] / view_users
).round(4)

# Calculate stage-to-stage conversion
funnel_summary["previous_stage_users"] = funnel_summary["unique_users"].shift(1)
funnel_summary["stage_to_stage_conversion"] = (
    funnel_summary["unique_users"] / funnel_summary["previous_stage_users"]
).round(4)

funnel_summary.loc[
    funnel_summary["event_type"] == "view", 
    "stage_to_stage_conversion"
] = 1.0

funnel_summary

,event_type,unique_users,conversion_from_view,previous_stage_users,stage_to_stage_conversion
0,view,93358,1.0000,NaN,1.0000
1,click,65258,0.6990,93358.0,0.6990
2,add_to_cart,48823,0.5230,65258.0,0.7482
3,inquiry,35024,0.3752,48823.0,0.7174
4,purchase,93358,1.0000,35024.0,2.6655


In [22]:
# Export funnel summary
funnel_summary.to_csv(TABLE_OUTPUT_DIR / "funnel_summary.csv", index=False)

print("funnel_summary.csv exported successfully.")

funnel_summary.csv exported successfully.


## Step 12: Category-Level Funnel Performance

This analysis compares funnel performance across product categories.

For each category, I calculate:

- View users
- Click users
- Inquiry users
- Purchase users
- Click-through rate
- Inquiry rate
- Purchase rate
- Total GMV

This helps identify which categories generate stronger lead quality and commercial outcomes.

In [24]:
# Pivot event counts by category
category_funnel = (
    fact_user_events
    .groupby(["category", "event_type"])["user_id"]
    .nunique()
    .reset_index()
    .pivot(index="category", columns="event_type", values="user_id")
    .fillna(0)
    .reset_index()
)

# Ensure all funnel columns exist
for col in funnel_order:
    if col not in category_funnel.columns:
        category_funnel[col] = 0

# Calculate category-level funnel rates
category_funnel["click_through_rate"] = (
    category_funnel["click"] / category_funnel["view"]
).replace([np.inf, -np.inf], 0).fillna(0).round(4)

category_funnel["inquiry_rate"] = (
    category_funnel["inquiry"] / category_funnel["view"]
).replace([np.inf, -np.inf], 0).fillna(0).round(4)

category_funnel["purchase_rate"] = (
    category_funnel["purchase"] / category_funnel["view"]
).replace([np.inf, -np.inf], 0).fillna(0).round(4)

# Add GMV by category from purchase events
category_gmv = (
    fact_user_events[fact_user_events["event_type"] == "purchase"]
    .groupby("category")["gmv"]
    .sum()
    .reset_index()
    .rename(columns={"gmv": "total_gmv"})
)

category_funnel = category_funnel.merge(category_gmv, on="category", how="left")
category_funnel["total_gmv"] = category_funnel["total_gmv"].fillna(0).round(2)

# Keep categories with enough views for reliable analysis
category_funnel = category_funnel[category_funnel["view"] >= 50]

category_funnel = category_funnel.sort_values(
    by="total_gmv", 
    ascending=False
)

category_funnel.head(15)

,category,add_to_cart,click,inquiry,purchase,view,click_through_rate,inquiry_rate,purchase_rate,total_gmv
43,health_beauty,4414,5959,3209,8498,8498,0.7012,0.3776,1.0,1412089.53
71,watches_gifts,2659,3693,1973,5421,5421,0.6812,0.3640,1.0,1264333.12
7,bed_bath_table,4627,6176,3403,9008,9008,0.6856,0.3778,1.0,1225209.26
65,sports_leisure,3886,5207,2750,7341,7341,0.7093,0.3746,1.0,1118256.91
15,computers_accessories,3322,4464,2401,6405,6405,0.6970,0.3749,1.0,1032723.77
39,furniture_decor,3356,4396,2426,6178,6178,0.7116,0.3927,1.0,880329.92
49,housewares,3018,4021,2159,5681,5681,0.7078,0.3800,1.0,758392.25
20,cool_stuff,1824,2476,1208,3543,3543,0.6988,0.3410,1.0,691680.89
5,auto,1905,2581,1417,3769,3769,0.6848,0.3760,1.0,669454.75
42,garden_tools,1836,2401,1346,3420,3420,0.7020,0.3936,1.0,567145.68


In [25]:
# Export category funnel performance
category_funnel.to_csv(TABLE_OUTPUT_DIR / "category_funnel_performance.csv", index=False)

print("category_funnel_performance.csv exported successfully.")

category_funnel_performance.csv exported successfully.


## Step 13: Traffic Source Funnel Analysis

Traffic source analysis helps evaluate which acquisition channels bring stronger engagement and conversion.

In a TikTok LIVE context, possible sources include:

- For You Feed
- Live Stream
- Search
- Shop Tab
- Creator Profile

This analysis can help product and recommendation teams understand where high-intent leads are coming from.

In [26]:
# Pivot event counts by traffic source
traffic_funnel = (
    fact_user_events
    .groupby(["traffic_source", "event_type"])["user_id"]
    .nunique()
    .reset_index()
    .pivot(index="traffic_source", columns="event_type", values="user_id")
    .fillna(0)
    .reset_index()
)

# Ensure all funnel columns exist
for col in funnel_order:
    if col not in traffic_funnel.columns:
        traffic_funnel[col] = 0

# Calculate funnel rates
traffic_funnel["click_through_rate"] = (
    traffic_funnel["click"] / traffic_funnel["view"]
).replace([np.inf, -np.inf], 0).fillna(0).round(4)

traffic_funnel["inquiry_rate"] = (
    traffic_funnel["inquiry"] / traffic_funnel["view"]
).replace([np.inf, -np.inf], 0).fillna(0).round(4)

traffic_funnel["purchase_rate"] = (
    traffic_funnel["purchase"] / traffic_funnel["view"]
).replace([np.inf, -np.inf], 0).fillna(0).round(4)

# Add GMV by traffic source
traffic_gmv = (
    fact_user_events[fact_user_events["event_type"] == "purchase"]
    .groupby("traffic_source")["gmv"]
    .sum()
    .reset_index()
    .rename(columns={"gmv": "total_gmv"})
)

traffic_funnel = traffic_funnel.merge(traffic_gmv, on="traffic_source", how="left")
traffic_funnel["total_gmv"] = traffic_funnel["total_gmv"].fillna(0).round(2)

traffic_funnel = traffic_funnel.sort_values(by="purchase_rate", ascending=False)

traffic_funnel

,traffic_source,add_to_cart,click,inquiry,purchase,view,click_through_rate,inquiry_rate,purchase_rate,total_gmv
0,Creator Profile,4168,5885,2992,8641,8641,0.6811,0.3463,1.0,1208962.30
1,For You Feed,18246,24523,12946,36004,36004,0.6811,0.3596,1.0,5463458.12
2,Live Stream,13104,17855,9169,26012,26012,0.6864,0.3525,1.0,3784623.23
3,Search,9407,12863,6701,19031,19031,0.6759,0.3521,1.0,2793640.34
4,Shop Tab,7472,10147,5189,14935,14935,0.6794,0.3474,1.0,2169089.76


In [27]:
# Export traffic source funnel performance
traffic_funnel.to_csv(TABLE_OUTPUT_DIR / "traffic_source_funnel.csv", index=False)

print("traffic_source_funnel.csv exported successfully.")

traffic_source_funnel.csv exported successfully.


## Step 14: User Segment Funnel Analysis

This analysis compares conversion patterns across user value segments.

User segments:

- Low Value
- Medium Value
- High Value

This helps evaluate whether historically higher-value users show stronger engagement and conversion behaviour.

In [28]:
# Pivot event counts by user value segment
segment_funnel = (
    fact_user_events
    .groupby(["user_value_segment", "event_type"])["user_id"]
    .nunique()
    .reset_index()
    .pivot(index="user_value_segment", columns="event_type", values="user_id")
    .fillna(0)
    .reset_index()
)

# Ensure all funnel columns exist
for col in funnel_order:
    if col not in segment_funnel.columns:
        segment_funnel[col] = 0

# Calculate funnel rates
segment_funnel["click_through_rate"] = (
    segment_funnel["click"] / segment_funnel["view"]
).replace([np.inf, -np.inf], 0).fillna(0).round(4)

segment_funnel["inquiry_rate"] = (
    segment_funnel["inquiry"] / segment_funnel["view"]
).replace([np.inf, -np.inf], 0).fillna(0).round(4)

segment_funnel["purchase_rate"] = (
    segment_funnel["purchase"] / segment_funnel["view"]
).replace([np.inf, -np.inf], 0).fillna(0).round(4)

# Add GMV by user segment
segment_gmv = (
    fact_user_events[fact_user_events["event_type"] == "purchase"]
    .groupby("user_value_segment")["gmv"]
    .sum()
    .reset_index()
    .rename(columns={"gmv": "total_gmv"})
)

segment_funnel = segment_funnel.merge(segment_gmv, on="user_value_segment", how="left")
segment_funnel["total_gmv"] = segment_funnel["total_gmv"].fillna(0).round(2)

segment_funnel

,user_value_segment,add_to_cart,click,inquiry,purchase,view,click_through_rate,inquiry_rate,purchase_rate,total_gmv
0,High Value,17998,23387,13090,31118,31118,0.7516,0.4207,1.0,10454977.90
1,Low Value,14912,20513,10584,31120,31120,0.6592,0.3401,1.0,1552691.00
2,Medium Value,15913,21358,11350,31120,31120,0.6863,0.3647,1.0,3412104.85


In [29]:
# Export user segment funnel performance
segment_funnel.to_csv(TABLE_OUTPUT_DIR / "user_segment_funnel.csv", index=False)

print("user_segment_funnel.csv exported successfully.")

user_segment_funnel.csv exported successfully.


## Step 15: Seller Lead Quality Analysis

Seller lead quality analysis evaluates how effectively each seller converts user interest into purchases.

For each seller, I calculate:

- View users
- Inquiry users
- Purchase users
- Inquiry rate
- Purchase rate
- Total GMV
- Average review score

This can support seller-side growth insights and recommendation strategy decisions.

In [30]:
# Pivot event counts by seller
seller_funnel = (
    fact_user_events
    .groupby(["seller_id", "event_type"])["user_id"]
    .nunique()
    .reset_index()
    .pivot(index="seller_id", columns="event_type", values="user_id")
    .fillna(0)
    .reset_index()
)

# Ensure all funnel columns exist
for col in funnel_order:
    if col not in seller_funnel.columns:
        seller_funnel[col] = 0

# Calculate seller funnel rates
seller_funnel["inquiry_rate"] = (
    seller_funnel["inquiry"] / seller_funnel["view"]
).replace([np.inf, -np.inf], 0).fillna(0).round(4)

seller_funnel["purchase_rate"] = (
    seller_funnel["purchase"] / seller_funnel["view"]
).replace([np.inf, -np.inf], 0).fillna(0).round(4)

# Add seller GMV and review score
seller_quality = (
    fact_user_events[fact_user_events["event_type"] == "purchase"]
    .groupby("seller_id")
    .agg(
        total_gmv=("gmv", "sum"),
        avg_review_score=("review_score", "mean"),
        product_count=("product_id", "nunique")
    )
    .reset_index()
)

seller_funnel = seller_funnel.merge(seller_quality, on="seller_id", how="left")

seller_funnel["total_gmv"] = seller_funnel["total_gmv"].fillna(0).round(2)
seller_funnel["avg_review_score"] = seller_funnel["avg_review_score"].round(2)

# Keep sellers with enough views
seller_funnel = seller_funnel[seller_funnel["view"] >= 20]

seller_funnel = seller_funnel.sort_values(
    by=["purchase_rate", "total_gmv"], 
    ascending=[False, False]
)

seller_funnel.head(20)

,seller_id,add_to_cart,click,inquiry,purchase,view,inquiry_rate,purchase_rate,total_gmv,avg_review_score,product_count
834,4869f7a5dfa277a7dca6462dcf3b52b2,598.0,774.0,402.0,1117.0,1117.0,0.3599,1.0,247007.06,4.15,94
1480,7c67e1448b00f6e969d365cea6b010ab,494.0,655.0,330.0,962.0,962.0,0.3430,1.0,237806.69,3.36,197
858,4a3ca9315b744ce9f8e9374361493884,816.0,1119.0,621.0,1759.0,1759.0,0.3530,1.0,231220.43,3.84,394
982,53243585a1d6dc2643021fd1853d8905,203.0,260.0,127.0,340.0,340.0,0.3735,1.0,230797.02,4.13,23
2903,fa1c13f2614d7b5c4749cbc52fecda94,304.0,419.0,193.0,575.0,575.0,0.3357,1.0,200833.50,4.38,284
2543,da8622b14eb17ae2831f4ac5b9dab84a,718.0,922.0,536.0,1272.0,1272.0,0.4214,1.0,184706.78,4.07,221
1504,7e93a43ef30c4f03f38b393420bc753a,156.0,222.0,114.0,319.0,319.0,0.3574,1.0,171973.55,4.37,175
188,1025f0e2d44d7041d6cf58b6550e0bfa,521.0,681.0,404.0,894.0,894.0,0.4519,1.0,171924.96,3.88,153
1450,7a67c85e85bb2ce8582c35f2203ad736,578.0,780.0,396.0,1140.0,1140.0,0.3474,1.0,160278.52,4.27,146
1758,955fee9216a65b617aa5c0531780ce60,669.0,919.0,456.0,1256.0,1256.0,0.3631,1.0,156606.48,4.09,103


In [31]:
# Export seller lead quality analysis
seller_funnel.to_csv(TABLE_OUTPUT_DIR / "seller_lead_quality.csv", index=False)

print("seller_lead_quality.csv exported successfully.")

seller_lead_quality.csv exported successfully.


## Step 16: Create Daily Funnel Trend

Daily funnel trend helps monitor how user engagement and conversion change over time.

This output can later be used in Tableau to build time-series charts for product performance monitoring.

In [32]:
# Create event date
fact_user_events["event_date"] = fact_user_events["event_time"].dt.date

# Daily event trend
daily_funnel_trend = (
    fact_user_events
    .groupby(["event_date", "event_type"])["user_id"]
    .nunique()
    .reset_index()
    .pivot(index="event_date", columns="event_type", values="user_id")
    .fillna(0)
    .reset_index()
)

# Ensure all funnel columns exist
for col in funnel_order:
    if col not in daily_funnel_trend.columns:
        daily_funnel_trend[col] = 0

daily_funnel_trend["click_through_rate"] = (
    daily_funnel_trend["click"] / daily_funnel_trend["view"]
).replace([np.inf, -np.inf], 0).fillna(0).round(4)

daily_funnel_trend["inquiry_rate"] = (
    daily_funnel_trend["inquiry"] / daily_funnel_trend["view"]
).replace([np.inf, -np.inf], 0).fillna(0).round(4)

daily_funnel_trend["purchase_rate"] = (
    daily_funnel_trend["purchase"] / daily_funnel_trend["view"]
).replace([np.inf, -np.inf], 0).fillna(0).round(4)

daily_funnel_trend.head()

event_type,event_date,add_to_cart,click,inquiry,purchase,view,click_through_rate,inquiry_rate,purchase_rate
0,2016-09-15,1.0,1.0,1.0,1.0,1.0,1.0000,1.0000,1.0000
1,2016-10-03,4.0,5.0,2.0,7.0,7.0,0.7143,0.2857,1.0000
2,2016-10-04,33.0,34.0,25.0,54.0,55.0,0.6182,0.4545,0.9818
3,2016-10-05,17.0,24.0,12.0,35.0,35.0,0.6857,0.3429,1.0000
4,2016-10-06,28.0,31.0,16.0,41.0,42.0,0.7381,0.3810,0.9762


In [33]:
# Export daily funnel trend
daily_funnel_trend.to_csv(TABLE_OUTPUT_DIR / "daily_funnel_trend.csv", index=False)

print("daily_funnel_trend.csv exported successfully.")

daily_funnel_trend.csv exported successfully.


In [34]:
# Export simulated user event log
fact_user_events.to_csv(FINAL_DIR / "fact_user_events.csv", index=False)

print("fact_user_events.csv exported successfully.")
print("File saved to:", FINAL_DIR / "fact_user_events.csv")

fact_user_events.csv exported successfully.
File saved to: ../data/final/fact_user_events.csv


## Step 17: Summary of User Funnel Analysis

In this notebook, I created a simulated user interaction event log based on real delivered orders from the Olist dataset.

Key outputs generated:

- `fact_user_events.csv`
- `funnel_summary.csv`
- `category_funnel_performance.csv`
- `traffic_source_funnel.csv`
- `user_segment_funnel.csv`
- `seller_lead_quality.csv`
- `daily_funnel_trend.csv`

Business insights supported by this stage:

1. Identify where users drop off in the lead generation funnel;
2. Compare conversion performance across product categories;
3. Evaluate traffic source quality;
4. Compare lead quality across user value segments;
5. Identify sellers with stronger user engagement and purchase conversion.

This stage creates the behavioural foundation for the next parts of the project:

- LLM-assisted purchase intent classification;
- Lead scoring;
- Intent-aware recommendation strategy;
- A/B test evaluation.